# 实验结果分析：Token 生成速度与阈值的关系

本 Notebook 用于分析 `experiment_summary_20260117_030023.json` 中的实验数据。
重点考察 `little_draft_threshold` (对应配置中的 `small_draft_threshold`) 和 `draft_target_threshold` 对 `throughput` (Token 生成速度) 的影响。

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 设置绘图风格
sns.set_theme(style="whitegrid")
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans'] # 尝试支持中文显示，如果没有 SimHei 会回退
plt.rcParams['axes.unicode_minus'] = False # 解决负号显示问题

In [ ]:
import os
print(f"Current working directory: {os.getcwd()}")
print(f"Files in current directory: {os.listdir('.')}")
print(f"Is file exists: {os.path.exists('experiment_summary_20260126_185941.json')}")

In [ ]:
# 读取数据
file_path = "vicuna_scan.json"

try:
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    print(f"成功读取 {len(data)} 条实验记录")
except FileNotFoundError:
    print(f"找不到文件: {file_path}")
    data = []

# 提取关键信息
extracted_data = []
for entry in data:
    config = entry.get("config", {})
    result = entry.get("result", {})
    
    # 提取阈值和吞吐量
    little_draft_thresh = config.get("small_draft_threshold")
    draft_target_thresh = config.get("draft_target_threshold")
    throughput = result.get("throughput")
    
    # 提取前向传播次数和计算时间
    little_fwd = result.get("little_forward_times")
    draft_fwd = result.get("draft_forward_times")
    target_fwd = result.get("target_forward_times")
    computation_time = result.get("computation_time")
    
    # 提取通信相关指标
    comm_time = result.get("communication_time")
    edge_cloud_data = result.get("edge_cloud_data_bytes")
    edge_end_data = result.get("edge_end_data_bytes")
    comm_energy = result.get("comm_energy")
    
    # 提取数据集名称
    eval_dataset = config.get("eval_dataset", "unknown")
    dataset_name = eval_dataset.split("/")[-1].replace("eval_", "").replace(".py", "").replace("_noeval", "")
    
    if little_draft_thresh is not None and draft_target_thresh is not None and throughput is not None:
        extracted_data.append({
            "dataset": dataset_name,
            "little_draft_threshold": float(little_draft_thresh),
            "draft_target_threshold": float(draft_target_thresh),
            "throughput": float(throughput),
            "little_forward_times": int(little_fwd) if little_fwd is not None else 0,
            "draft_forward_times": int(draft_fwd) if draft_fwd is not None else 0,
            "target_forward_times": int(target_fwd) if target_fwd is not None else 0,
            "computation_time": float(computation_time) if computation_time is not None else None,
            "comm_time": float(comm_time) if comm_time is not None else None,
            "edge_cloud_data_MB": float(edge_cloud_data) / (1024*1024) if edge_cloud_data is not None else None,
            "edge_end_data_MB": float(edge_end_data) / (1024*1024) if edge_end_data is not None else None,
            "comm_energy": float(comm_energy) if comm_energy is not None else None
        })

# 创建 DataFrame
df = pd.DataFrame(extracted_data)

# 显示前几行
if not df.empty:
    print(f"提取到 {len(df)} 条有效数据")
    display(df.head())
    display(df.describe())
else:
    print("提取到的数据为空")

In [ ]:
# 绘制折线图
plt.figure(figsize=(12, 6))

# 使用 hue 将 draft_target_threshold 分组
sns.lineplot(
    data=df, 
    x="little_draft_threshold", 
    y="throughput", 
    hue="draft_target_threshold",
    marker="o",
    palette="viridis"
)

plt.title("Token 生成速度 vs. Little Draft Threshold (按 Draft Target Threshold 分组)")
plt.xlabel("Little Draft Threshold (small_draft_threshold)")
plt.ylabel("Throughput (tokens/s)")
plt.legend(title="Draft Target Threshold")
plt.grid(True)
plt.show()

# 此外，也可以反过来画，看 Draft Target Threshold 的影响
plt.figure(figsize=(12, 6))

sns.lineplot(
    data=df, 
    x="draft_target_threshold", 
    y="throughput", 
    hue="little_draft_threshold",
    marker="s",
    palette="magma"
)

plt.title("Token 生成速度 vs. Draft Target Threshold (按 Little Draft Threshold 分组)")
plt.xlabel("Draft Target Threshold")
plt.ylabel("Throughput (tokens/s)")
plt.legend(title="Little Draft Threshold")
plt.grid(True)
plt.show()

In [ ]:
# 绘制热力图

# 1. 重塑数据：行=little_draft_threshold, 列=draft_target_threshold
# 如果同一对 (little_draft_threshold, draft_target_threshold) 有多条数据，默认取平均值
heatmap_data = df.pivot_table(
    index="little_draft_threshold", 
    columns="draft_target_threshold", 
    values="throughput",
    aggfunc='mean'
)

# 2. 绘制热力图
plt.figure(figsize=(10, 8))
sns.heatmap(
    heatmap_data, 
    annot=True, 
    fmt=".2f", 
    cmap="YlGnBu", 
    cbar_kws={'label': 'Throughput (tokens/s)'}
)

plt.title("Throughput Heatmap: Little Draft vs Draft Target Thresholds")
# 注意：Heatmap 的 y 轴通常是从上到下的，这里为了直观可能需要根据数值大小排序检查一下
# seaborn 默认会按 index 排序，通常是从小到大，但画在图上是从上到下增长还是减少取决于 seaborn 版本和设置
# 这里默认即可，注意观察坐标轴标签

plt.xlabel("Draft Target Threshold")
plt.ylabel("Little Draft Threshold")
plt.show()

In [ ]:
# --- 绘制 Target Forward Times 分析图 ---

# 绘制折线图
plt.figure(figsize=(12, 6))

# 使用 hue 将 draft_target_threshold 分组
sns.lineplot(
    data=df, 
    x="little_draft_threshold", 
    y="target_forward_times", 
    hue="draft_target_threshold",
    marker="o",
    palette="viridis"
)

plt.title("Target Forward Times vs. Little Draft Threshold (按 Draft Target Threshold 分组)")
plt.xlabel("Little Draft Threshold (small_draft_threshold)")
plt.ylabel("Target Forward Times")
plt.legend(title="Draft Target Threshold")
plt.grid(True)
plt.show()

# 反向分组
plt.figure(figsize=(12, 6))

sns.lineplot(
    data=df, 
    x="draft_target_threshold", 
    y="target_forward_times", 
    hue="little_draft_threshold",
    marker="s",
    palette="magma"
)

plt.title("Target Forward Times vs. Draft Target Threshold (按 Little Draft Threshold 分组)")
plt.xlabel("Draft Target Threshold")
plt.ylabel("Target Forward Times")
plt.legend(title="Little Draft Threshold")
plt.grid(True)
plt.show()

In [ ]:
# 绘制 Target Forward Times 热力图

# 1. 重塑数据
heatmap_data_forward = df.pivot_table(
    index="little_draft_threshold", 
    columns="draft_target_threshold", 
    values="target_forward_times",
    aggfunc='mean'
)

# 2. 绘制热力图
plt.figure(figsize=(10, 8))
sns.heatmap(
    heatmap_data_forward, 
    annot=True, 
    fmt=".0f",  # 次数一般是整数，用 .0f 显示
    cmap="YlOrRd", # 使用不同的颜色映射以区分
    cbar_kws={'label': 'Target Forward Times'}
)

plt.title("Target Forward Times Heatmap: Little Draft vs Draft Target Thresholds")
plt.xlabel("Draft Target Threshold")
plt.ylabel("Little Draft Threshold")
plt.show()

In [ ]:
# 找出每个数据集的最佳阈值组合（吞吐量最大），并展示全面指标
if not df.empty:
    # 按照 dataset 分组并找到 throughput 最大值的索引
    idx = df.groupby('dataset')['throughput'].idxmax()
    best_configs = df.loc[idx].copy()
    
    print("每个数据集的最佳阈值组合（吞吐量最大）及其详细指标：")
    columns_to_show = [
        'dataset', 
        'little_draft_threshold', 
        'draft_target_threshold', 
        'throughput', 
        'little_forward_times',
        'draft_forward_times',
        'target_forward_times',
        'computation_time',
        'comm_time', 
        'edge_cloud_data_MB', 
        'comm_energy'
    ]
    # 只显示存在的列
    columns_to_show = [c for c in columns_to_show if c in best_configs.columns]
    display(best_configs[columns_to_show])
else:
    print("没有可分析的数据。")